In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, 6)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

In [ ]:
x = np.genfromtxt("./data/nn_example_testdata1.csv", delimiter=",")
x_train = torch.FloatTensor(x)

y = np.asarray(["Methane", "n-Butane", "Propane", "CO2", "CO", "SF6"])
y_train_one_hot = np.eye(len(y))
y_train_one_hot = torch.FloatTensor(y_train_one_hot)

In [ ]:
model = MLP()
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
model.train()
epoch = 10000
loss_all = []
for ep in range(epoch):
    optimizer.zero_grad()
    # Forward pass
    y_pred = model(x_train)
    # Compute Loss
    loss = criterion(y_pred.squeeze(), y_train_one_hot)
    loss_all.append(loss.item())
    #     if epoch % 1000 == 0:
    #         print('Epoch {}: train loss: {}'.format(epoch, loss.item()))
    # Backward pass
    loss.backward()
    optimizer.step()


plt.plot(loss_all)
plt.show()

In [ ]:
model.eval()
y_pred = model(x_train)
print(y_pred)
y_pred = y_pred.detach().numpy()

y_pred_one_hot = np.zeros(y_pred.shape)  # our zeros and ones will go here
y_pred_one_hot[np.arange(y_pred.shape[0]), np.argmax(y_pred, axis=1)] = 1
print(y_pred_one_hot)

In [ ]:
while True:
    sub = input(
        "Pick a substance from below.\n 1. Methane\n 2. n-Butane\n 3. Propane\n 4. CO2\n 5. CO\n 6. SF6\n\n"
    )
    print("The signature of", y[int(sub) - 1], "is")
    print(x[int(sub) - 1, :])

    time.sleep(2)

    print("feeding to classifier...")
    y_pred = model(x_train[int(sub) - 1 : int(sub), :])
    y_pred = y_pred.detach().numpy()
    sub_ind = np.argmax(y_pred, axis=1)

    time.sleep(2)

    print("The substance is", y[sub_ind], "\n\n")

    time.sleep(3)